# 5. Comparison with BDHS 2017–18

This notebook is step 5 of 5. It applies the definitions and design-based methods of the 2022 analysis to the 2017–18 survey, for the comparison in Discussion 4.1 and Table SA3.

| Section | What it does |
|---|---|
| 1 | Sample and prevalence |
| 2 | Prevalence ratios |
| 3 | Inequality |
| 4 | Glucose distribution and the report's diabetes figure |
| 5 | Rounding pattern of the glucose variables |

**Files needed:** `BDIR7RFL.DTA` and `BDPR7RFL.DTA` in `data/raw/bdhs2017_18/`.

The 2017–18 recode has one glucose variable, `sb335b`, so the two scales coincide. Results go to `output/bdhs2017_18/` and `data/derived/bdhs2017_18/`, so no 2022 output is changed.

## Setup

The `bdhs2017_18` settings in `config/config.yml` point the same functions to the 2017–18 files and folders.

In [1]:
# Find the project folder, wherever the notebook was opened from
root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(root, "gdm-project.Rproj"))) {
  if (dirname(root) == root) stop("Open this notebook inside the gdm-project folder.")
  root <- dirname(root)
}
setwd(root)

# Use the package versions locked in renv.lock
if (!nzchar(Sys.getenv("RENV_PROJECT"))) {
  Sys.setenv(RENV_CONFIG_NAMESPACES_CHECK = "FALSE")
  source("renv/activate.R")
}
options(width = 120)

# Use the 2017-18 settings, then the shared functions
Sys.setenv(R_CONFIG_ACTIVE = "bdhs2017_18")
source(here::here("src", "common", "setup.R"))
Sys.unsetenv("R_CONFIG_ACTIVE")

# Stop early with a clear message if the 2017-18 files are not available
stopifnot("Place BDIR7RFL.DTA in data/raw/bdhs2017_18/" =
            file.exists(here::here("data", "raw", "bdhs2017_18", "BDIR7RFL.DTA")))

## 1. Sample and prevalence

The variables are renamed to their 2022 names on reading, so the 2022 derivations are reused unchanged.

Functions: `src/bdhs2017_18/bdhs2017.R`.

In [2]:
files <- read_and_merge_2017(cfg)
women <- derive_variables(files$women, cfg)
sample <- women[women$analytic, ]
saveRDS(women, data_file("women"))
saveRDS(sample, data_file("analytic_sample"))

flow <- participant_flow(women, cfg)
print(flow)
save_table(flow, "sample_flow")

                                         step n_remaining n_excluded
1                           Women interviewed       20127         NA
2                          Currently pregnant        1129      18998
3 In household selected for biomarker testing         316        813
4                      Aged 18 years or older         280         36
5          Valid fasting plasma glucose value         262         18
6   Fasted at least 8 hours (analytic sample)         262          0
7      No self-reported pre-existing diabetes         256          6
8       FPG below 7.0 mmol/L (outcome sample)         251          5
9      Complete covariate data (model sample)         251          0


In [3]:
des <- svy_domain(sample, women)

# Overall, and by trimester, residence and wealth
prev <- rbind(cbind(group = "all", level = "all", weighted_prevalence(des, "gdm")),
              do.call(rbind, lapply(c("trimester", "residence", "wealth"), function(v) prevalence_by(des, "gdm", v))))
print(prev)
save_table(prev, "prevalence")
save_table(cbind(outcome = "fpg_dm_range", weighted_prevalence(des, "fpg_dm_range")), "diabetes_range_pregnant")

      group   level   n events prevalence         se  ci_lower  ci_upper
1       all     all 251     91  0.3608750 0.03320107 0.2982543 0.4286129
2 trimester     1st  76     36  0.5003386 0.06112686 0.3799517 0.6206863
3 trimester     2nd  95     36  0.3853639 0.05470141 0.2833942 0.4985005
4 trimester     3rd  80     19  0.2021881 0.04746186 0.1233346 0.3134317
5 residence   Rural 159     47  0.2836831 0.03689946 0.2165607 0.3619969
6 residence   Urban  92     44  0.5511111 0.05728271 0.4358533 0.6611293
7    wealth  Lowest  85     27  0.2982877 0.05444323 0.2017739 0.4168579
8    wealth  Middle  85     26  0.2935124 0.05423705 0.1975372 0.4121673
9    wealth Highest  81     38  0.5077698 0.05977260 0.3893554 0.6253187


## 2. Prevalence ratios

The adjusted and minimal models, as in 2022.

In [4]:
x <- cfg$covariates
adjusted <- fit_pr(des, x, "gdm")
minimal <- do.call(rbind, lapply(c("wealth", "residence"), function(v) {
  e <- effect_table(fit_pr(des, c(v, "age_group", "trimester"), "gdm"), "PR", "minimal")
  e[startsWith(e$term, v), ]
}))
effects <- rbind(effect_table(adjusted, "PR", "adjusted"), minimal)
effects$design_df <- degf(adjusted$survey.design)
print(effects)
save_table(effects, "effects")

                   model measure           term  estimate  ci_lower  ci_upper      p_value   n design_df
age_group25+    adjusted      PR   age_group25+ 1.3498917 0.9461055 1.9260089 0.0975893200 251       179
residenceUrban  adjusted      PR residenceUrban 1.5981574 1.1093967 2.3022488 0.0121526051 251       179
wealthMiddle    adjusted      PR   wealthMiddle 0.8999728 0.5475322 1.4792758 0.6761077284 251       179
wealthHighest   adjusted      PR  wealthHighest 1.1639054 0.7133673 1.8989875 0.5414646108 251       179
trimester2nd    adjusted      PR   trimester2nd 0.7521449 0.5280512 1.0713394 0.1139205491 251       179
trimester3rd    adjusted      PR   trimester3rd 0.4531235 0.2666217 0.7700832 0.0036720438 251       179
bmi             adjusted      PR            bmi 1.0043260 0.9605548 1.0500917 0.8486300179 251       179
wealthMiddle1    minimal      PR   wealthMiddle 0.9417004 0.5674796 1.5626985 0.8152356981 251       179
wealthHighest1   minimal      PR  wealthHighest 1.45940

## 3. Inequality

Concentration indices with Rao–Wu bootstrap CIs.

In [5]:
boot <- bootstrap_weights(sample, women, cfg)
keep <- !is.na(sample$gdm)
d <- sample[keep, ]
est <- inequality_indices(d$gdm, d$wealth_score, d$weight)
reps <- bootstrap_indices(d$gdm, d$wealth_score, boot$weights[keep, ])
ineq <- cbind(index = c("relative", "wagstaff", "erreygers"), rbind(
  bootstrap_summary(est$C_relative, reps[1, ], boot),
  bootstrap_summary(est$C_wagstaff, reps[2, ], boot),
  bootstrap_summary(est$C_erreygers, reps[3, ], boot)))
print(ineq)
save_table(ineq, "inequality")

      index  estimate    boot_se   ci_lower  ci_upper    B failed
1  relative 0.1334668 0.05551102 0.02653026 0.2489562 1000      0
2  wagstaff 0.2088274 0.08615315 0.04014219 0.3771770 1000      0
3 erreygers 0.1926594 0.07958927 0.03713528 0.3397902 1000      0


## 4. Glucose distribution and the report's diabetes figure

Glucose in pregnant and non-pregnant women, and diabetes among women aged 18 or older, which the 2017–18 survey report gives as 10%.

In [6]:
dist <- fpg_distribution(women, "fpg")
print(dist)
save_table(dist, "fpg_distribution")

dm <- diabetes_2017(files$pr)
print(dm)
save_table(dm, "diabetes_report_replication")

  scale                        group    n q25 median q75 pct_fpg_5.1_or_more    se_5.1 pct_fpg_7.0_or_more    se_7.0
1   fpg                     Pregnant  262 4.4    4.8 5.4            39.14948 3.2525483            2.730559 1.1983764
2   fpg                 Non-pregnant 4581 5.0    5.4 6.0            72.02836 0.8983253            7.865636 0.5109857
3   fpg Non-pregnant, same age range 3589 5.0    5.4 5.9            71.33828 1.0016953            6.786782 0.5369231
     n diabetes report_2017_18
1 6971 9.520091             10


## 5. Rounding pattern

Whole mg/dL readings converted with a factor leave a repeating pattern in the counts per tenth of a mmol/L. The two 2022 variables have known factors (1.11 and 1.00), so they act as controls for the 2017–18 variable.

Lower deviance means a better fit. The 2022 file is read for two columns only.

In [7]:
p22 <- zap_labels(read_dta(here::here("data", "raw", "bdhs2022", "BDPR81FL.DTA"), col_select = dplyr::all_of(c("sb267", "sb267g"))))
ok22 <- !p22$sb267 %in% c(NA, 994:996) & !p22$sb267g %in% c(NA, 994:996)
tenths17 <- round(files$pr$sb267g[!is.na(files$pr$sb267g)] * 10)

sig <- rbind(rounding_signature(round(p22$sb267g[ok22] * 10), "2022 sb267g (known factor 1.11)"),
             rounding_signature(round(p22$sb267[ok22] / 18 * 10), "2022 sb267/18 (known factor 1.00)"),
             rounding_signature(tenths17, "2017-18 sb335b, all adults tested"))
print(sig)
save_table(sig, "rounding_signature")

                           variable     n deviance_no_pattern deviance_factor_1.00 deviance_factor_1.11 best_factor df
1   2022 sb267g (known factor 1.11)  7357           763.51566           1219.09426             53.76119        1.11 47
2 2022 sb267/18 (known factor 1.00)  7409           490.92914             56.63593            943.72280        1.00 47
3 2017-18 sb335b, all adults tested 11853            73.31676            741.74213           1096.81847          NA 47
  n_unreachable_tenths n_at_unreachable_tenths
1                    6                     442
2                    6                     477
3                    6                    1094


**Done.** All results are in `output/bdhs2022/` and `output/bdhs2017_18/`.